🚕 Predicting Uber Ride Cancellations

Domain: Urban Mobility & Ride-Hailing Analytics

🎯 Objective

Build a predictive model to determine whether a customer will cancel a ride before it begins, using only the booking metadata available at the time of booking.
The goal is to help the platform proactively identify high-risk cancellations and optimize driver dispatch efficiency.

In [715]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)


In [716]:
df = pd.read_csv('/Users/shubh/Downloads/Capstone/Git/data/ncr_ride_bookings.csv')

In [717]:
df.head()

,Date,Time,Booking ID,Booking Status,Customer ID,Vehicle Type,Pickup Location,Drop Location,Avg VTAT,Avg CTAT,Cancelled Rides by Customer,Reason for cancelling by Customer,Cancelled Rides by Driver,Driver Cancellation Reason,Incomplete Rides,Incomplete Rides Reason,Booking Value,Ride Distance,Driver Ratings,Customer Rating,Payment Method
0,2024-03-23,12:29:38,"""CNR5884300""",No Driver Found,"""CID1982111""",eBike,Palam Vihar,Jhilmil,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2024-11-29,18:01:39,"""CNR1326809""",Incomplete,"""CID4604802""",Go Sedan,Shastri Nagar,Gurgaon Sector 56,4.9,14.0,NaN,NaN,NaN,NaN,1.0,Vehicle Breakdown,237.0,5.73,NaN,NaN,UPI
2,2024-08-23,08:56:10,"""CNR8494506""",Completed,"""CID9202816""",Auto,Khandsa,Malviya Nagar,13.4,25.8,NaN,NaN,NaN,NaN,NaN,NaN,627.0,13.58,4.9,4.9,Debit Card
3,2024-10-21,17:17:25,"""CNR8906825""",Completed,"""CID2610914""",Premier Sedan,Central Secretariat,Inderlok,13.1,28.5,NaN,NaN,NaN,NaN,NaN,NaN,416.0,34.02,4.6,5.0,UPI
4,2024-09-16,22:08:00,"""CNR1950162""",Completed,"""CID9933542""",Bike,Ghitorni Village,Khan Market,5.3,19.6,NaN,NaN,NaN,NaN,NaN,NaN,737.0,48.21,4.1,4.3,UPI


In [718]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 150000 entries, 0 to 149999
Data columns (total 21 columns):
 #   Column                             Non-Null Count   Dtype  
---  ------                             --------------   -----  
 0   Date                               150000 non-null  object 
 1   Time                               150000 non-null  object 
 2   Booking ID                         150000 non-null  object 
 3   Booking Status                     150000 non-null  object 
 4   Customer ID                        150000 non-null  object 
 5   Vehicle Type                       150000 non-null  object 
 6   Pickup Location                    150000 non-null  object 
 7   Drop Location                      150000 non-null  object 
 8   Avg VTAT                           139500 non-null  float64
 9   Avg CTAT                           102000 non-null  float64
 10  Cancelled Rides by Customer        10500 non-null   float64
 11  Reason for cancelling by Customer  1050

Avg VTAT:	Average time for driver to reach pickup location (in minutes)

Avg CTAT:	Average trip duration from pickup to destination (in minutes)

In [719]:
# Combine 'Date' and 'Time' into a single datetime column
df['datetime'] = pd.to_datetime(df['Date'] + ' ' + df['Time'], format='%Y-%m-%d %H:%M:%S', errors='coerce')

# Drop original columns if not needed
df.drop(['Date', 'Time'], axis=1, inplace=True)

In [720]:
# Verify the change
print(df.info())
print(df[['datetime']].head())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 150000 entries, 0 to 149999
Data columns (total 20 columns):
 #   Column                             Non-Null Count   Dtype         
---  ------                             --------------   -----         
 0   Booking ID                         150000 non-null  object        
 1   Booking Status                     150000 non-null  object        
 2   Customer ID                        150000 non-null  object        
 3   Vehicle Type                       150000 non-null  object        
 4   Pickup Location                    150000 non-null  object        
 5   Drop Location                      150000 non-null  object        
 6   Avg VTAT                           139500 non-null  float64       
 7   Avg CTAT                           102000 non-null  float64       
 8   Cancelled Rides by Customer        10500 non-null   float64       
 9   Reason for cancelling by Customer  10500 non-null   object        
 10  Cancelled Rides by D

In [721]:
df.head()

,Booking ID,Booking Status,Customer ID,Vehicle Type,Pickup Location,Drop Location,Avg VTAT,Avg CTAT,Cancelled Rides by Customer,Reason for cancelling by Customer,Cancelled Rides by Driver,Driver Cancellation Reason,Incomplete Rides,Incomplete Rides Reason,Booking Value,Ride Distance,Driver Ratings,Customer Rating,Payment Method,datetime
0,"""CNR5884300""",No Driver Found,"""CID1982111""",eBike,Palam Vihar,Jhilmil,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2024-03-23 12:29:38
1,"""CNR1326809""",Incomplete,"""CID4604802""",Go Sedan,Shastri Nagar,Gurgaon Sector 56,4.9,14.0,NaN,NaN,NaN,NaN,1.0,Vehicle Breakdown,237.0,5.73,NaN,NaN,UPI,2024-11-29 18:01:39
2,"""CNR8494506""",Completed,"""CID9202816""",Auto,Khandsa,Malviya Nagar,13.4,25.8,NaN,NaN,NaN,NaN,NaN,NaN,627.0,13.58,4.9,4.9,Debit Card,2024-08-23 08:56:10
3,"""CNR8906825""",Completed,"""CID2610914""",Premier Sedan,Central Secretariat,Inderlok,13.1,28.5,NaN,NaN,NaN,NaN,NaN,NaN,416.0,34.02,4.6,5.0,UPI,2024-10-21 17:17:25
4,"""CNR1950162""",Completed,"""CID9933542""",Bike,Ghitorni Village,Khan Market,5.3,19.6,NaN,NaN,NaN,NaN,NaN,NaN,737.0,48.21,4.1,4.3,UPI,2024-09-16 22:08:00


In [722]:
df['hour'] = df['datetime'].dt.hour
df['day_of_week'] = df['datetime'].dt.day_name()
df['is_weekend'] = df['day_of_week'].isin(['Saturday', 'Sunday'])

In [723]:
df.head()

,Booking ID,Booking Status,Customer ID,Vehicle Type,Pickup Location,Drop Location,Avg VTAT,Avg CTAT,Cancelled Rides by Customer,Reason for cancelling by Customer,Cancelled Rides by Driver,Driver Cancellation Reason,Incomplete Rides,Incomplete Rides Reason,Booking Value,Ride Distance,Driver Ratings,Customer Rating,Payment Method,datetime,hour,day_of_week,is_weekend
0,"""CNR5884300""",No Driver Found,"""CID1982111""",eBike,Palam Vihar,Jhilmil,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2024-03-23 12:29:38,12,Saturday,True
1,"""CNR1326809""",Incomplete,"""CID4604802""",Go Sedan,Shastri Nagar,Gurgaon Sector 56,4.9,14.0,NaN,NaN,NaN,NaN,1.0,Vehicle Breakdown,237.0,5.73,NaN,NaN,UPI,2024-11-29 18:01:39,18,Friday,False
2,"""CNR8494506""",Completed,"""CID9202816""",Auto,Khandsa,Malviya Nagar,13.4,25.8,NaN,NaN,NaN,NaN,NaN,NaN,627.0,13.58,4.9,4.9,Debit Card,2024-08-23 08:56:10,8,Friday,False
3,"""CNR8906825""",Completed,"""CID2610914""",Premier Sedan,Central Secretariat,Inderlok,13.1,28.5,NaN,NaN,NaN,NaN,NaN,NaN,416.0,34.02,4.6,5.0,UPI,2024-10-21 17:17:25,17,Monday,False
4,"""CNR1950162""",Completed,"""CID9933542""",Bike,Ghitorni Village,Khan Market,5.3,19.6,NaN,NaN,NaN,NaN,NaN,NaN,737.0,48.21,4.1,4.3,UPI,2024-09-16 22:08:00,22,Monday,False


In [724]:
df['Vehicle Type'].unique()

array(['eBike', 'Go Sedan', 'Auto', 'Premier Sedan', 'Bike', 'Go Mini',
       'Uber XL'], dtype=object)

In [725]:
df['Booking Status'].unique()

array(['No Driver Found', 'Incomplete', 'Completed',
       'Cancelled by Driver', 'Cancelled by Customer'], dtype=object)

In [726]:
df[df['Avg VTAT'].isnull()].head()

,Booking ID,Booking Status,Customer ID,Vehicle Type,Pickup Location,Drop Location,Avg VTAT,Avg CTAT,Cancelled Rides by Customer,Reason for cancelling by Customer,Cancelled Rides by Driver,Driver Cancellation Reason,Incomplete Rides,Incomplete Rides Reason,Booking Value,Ride Distance,Driver Ratings,Customer Rating,Payment Method,datetime,hour,day_of_week,is_weekend
0,"""CNR5884300""",No Driver Found,"""CID1982111""",eBike,Palam Vihar,Jhilmil,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2024-03-23 12:29:38,12,Saturday,True
8,"""CNR4510807""",No Driver Found,"""CID7873618""",Go Sedan,Noida Sector 62,Noida Sector 18,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2024-09-14 12:49:09,12,Saturday,True
11,"""CNR9551927""",No Driver Found,"""CID7568143""",Auto,Vidhan Sabha,AIIMS,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2024-09-18 08:09:38,8,Wednesday,False
27,"""CNR4499383""",No Driver Found,"""CID5717521""",Premier Sedan,Sadar Bazar Gurgaon,Mehrauli,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2024-04-12 19:42:35,19,Friday,False
57,"""CNR9773309""",No Driver Found,"""CID9965847""",Uber XL,Anand Vihar ISBT,Dwarka Sector 21,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2024-04-11 15:43:34,15,Thursday,False


In [727]:
df[df['Avg VTAT'].isnull()]['Booking Status'].unique()

array(['No Driver Found'], dtype=object)

In [728]:
df['Booking Status'].unique()

array(['No Driver Found', 'Incomplete', 'Completed',
       'Cancelled by Driver', 'Cancelled by Customer'], dtype=object)

In [729]:
df['Reason for cancelling by Customer'].unique() 

array([nan, 'Driver is not moving towards pickup location',
       'Driver asked to cancel', 'AC is not working', 'Change of plans',
       'Wrong Address'], dtype=object)

In [730]:
df[df['Booking Status']=='Incomplete'].head()

,Booking ID,Booking Status,Customer ID,Vehicle Type,Pickup Location,Drop Location,Avg VTAT,Avg CTAT,Cancelled Rides by Customer,Reason for cancelling by Customer,Cancelled Rides by Driver,Driver Cancellation Reason,Incomplete Rides,Incomplete Rides Reason,Booking Value,Ride Distance,Driver Ratings,Customer Rating,Payment Method,datetime,hour,day_of_week,is_weekend
1,"""CNR1326809""",Incomplete,"""CID4604802""",Go Sedan,Shastri Nagar,Gurgaon Sector 56,4.9,14.0,NaN,NaN,NaN,NaN,1.0,Vehicle Breakdown,237.0,5.73,NaN,NaN,UPI,2024-11-29 18:01:39,18,Friday,False
9,"""CNR7721892""",Incomplete,"""CID5214275""",Auto,Rohini,Adarsh Nagar,6.1,26.0,NaN,NaN,NaN,NaN,1.0,Other Issue,135.0,10.36,NaN,NaN,Cash,2024-12-16 19:06:48,19,Monday,False
28,"""CNR9834201""",Incomplete,"""CID7891331""",Bike,Shastri Park,Botanical Garden,3.1,28.2,NaN,NaN,NaN,NaN,1.0,Vehicle Breakdown,304.0,1.98,NaN,NaN,UPI,2024-04-05 18:57:12,18,Friday,False
42,"""CNR4130953""",Incomplete,"""CID3196782""",Premier Sedan,Panipat,Tagore Garden,6.4,10.5,NaN,NaN,NaN,NaN,1.0,Other Issue,966.0,18.30,NaN,NaN,Debit Card,2024-02-12 19:42:39,19,Monday,False
47,"""CNR4674790""",Incomplete,"""CID4795905""",Bike,Dilshad Garden,Nehru Place,2.3,29.7,NaN,NaN,NaN,NaN,1.0,Vehicle Breakdown,453.0,19.24,NaN,NaN,Debit Card,2024-09-30 13:10:50,13,Monday,False


In [731]:
df[df['Booking Status']=='Incomplete']['Incomplete Rides Reason'].unique()

array(['Vehicle Breakdown', 'Other Issue', 'Customer Demand'],
      dtype=object)

In [732]:
df[df['Cancelled Rides by Customer']==1]['Booking Status'].unique()

array(['Cancelled by Customer'], dtype=object)

In [733]:
# Cancelled Rides by Customer vs Booking Status 
pd.crosstab(df['Booking Status'], df['Cancelled Rides by Customer'], dropna=False)

Cancelled Rides by Customer,1.0,NaN
Booking Status,,
Cancelled by Customer,10500,0
Cancelled by Driver,0,27000
Completed,0,93000
Incomplete,0,9000
No Driver Found,0,10500


In [734]:
# Cancelled Rides by Driver vs Booking Status 
pd.crosstab(df['Booking Status'], df['Cancelled Rides by Driver'], dropna=False)

Cancelled Rides by Driver,1.0,NaN
Booking Status,,
Cancelled by Customer,0,10500
Cancelled by Driver,27000,0
Completed,0,93000
Incomplete,0,9000
No Driver Found,0,10500


In [735]:
# filling NaN as O "not cancelled"
df['Cancelled Rides by Customer'] = df['Cancelled Rides by Customer'].fillna(0).astype(int)
df['Cancelled Rides by Driver'] = df['Cancelled Rides by Driver'].fillna(0).astype(int)
df['Incomplete Rides'] = df['Incomplete Rides'].fillna(0).astype(int)

In [736]:
# Create binary flags from cancellation/incomplete columns
#df['is_cancelled_customer'] = df['Cancelled Rides by Customer'].notnull()
#df['is_cancelled_driver'] = df['Cancelled Rides by Driver'].notnull()
#df['is_incomplete'] = df['Incomplete Rides'].notnull()

# Create a flag for missing ratings and booking values
df['missing_driver_rating'] = df['Driver Ratings'].isnull().astype(int)
df['missing_customer_rating'] = df['Customer Rating'].isnull().astype(int)
df['missing_booking_value'] = df['Booking Value'].isnull().astype(int)
df['missing_payment_method'] = df['Payment Method'].isnull().astype(int)

In [737]:
df['Cancelled Rides by Customer'].unique()

array([0, 1])

In [738]:
df['Cancelled Rides by Driver'].unique()

array([0, 1])

In [739]:
df['Booking ID'].nunique()

148767

In [740]:
len(df)

150000

In [741]:
print(df['Booking ID'].isnull().sum())
print(df['Customer ID'].isnull().sum())

0
0


In [742]:
dup_mask = df['Booking ID'].duplicated(keep='first')
print("duplicate rows (excluding first):", dup_mask.sum())
df[dup_mask].head()


duplicate rows (excluding first): 1233


,Booking ID,Booking Status,Customer ID,Vehicle Type,Pickup Location,Drop Location,Avg VTAT,Avg CTAT,Cancelled Rides by Customer,Reason for cancelling by Customer,Cancelled Rides by Driver,Driver Cancellation Reason,Incomplete Rides,Incomplete Rides Reason,Booking Value,Ride Distance,Driver Ratings,Customer Rating,Payment Method,datetime,hour,day_of_week,is_weekend,missing_driver_rating,missing_customer_rating,missing_booking_value,missing_payment_method
5522,"""CNR5071968""",No Driver Found,"""CID6309096""",Auto,Kanhaiya Nagar,India Gate,NaN,NaN,0,NaN,0,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,2024-03-10 19:55:06,19,Sunday,True,1,1,1,1
7762,"""CNR8512595""",Completed,"""CID9741888""",Go Mini,Narsinghpur,Huda City Centre,3.5,15.4,0,NaN,0,NaN,0,NaN,187.0,41.45,4.4,4.2,Credit Card,2024-03-01 11:55:56,11,Friday,False,0,0,0,0
9587,"""CNR1029172""",Completed,"""CID6382731""",Auto,Inderlok,Laxmi Nagar,6.9,34.4,0,NaN,0,NaN,0,NaN,332.0,36.38,4.3,4.3,UPI,2024-12-17 19:19:02,19,Tuesday,False,0,0,0,0
9726,"""CNR7132372""",Completed,"""CID6950827""",Go Sedan,Kalkaji,Sushant Lok,3.8,18.3,0,NaN,0,NaN,0,NaN,389.0,44.95,3.7,4.2,UPI,2024-05-23 20:44:27,20,Thursday,False,0,0,0,0
10186,"""CNR7768664""",Completed,"""CID4473762""",eBike,Anand Vihar ISBT,Netaji Subhash Place,5.1,42.1,0,NaN,0,NaN,0,NaN,357.0,17.03,4.2,4.2,UPI,2024-12-14 21:15:59,21,Saturday,True,0,0,0,0


In [743]:
df[df['Booking ID']=='"CNR5071968"']

,Booking ID,Booking Status,Customer ID,Vehicle Type,Pickup Location,Drop Location,Avg VTAT,Avg CTAT,Cancelled Rides by Customer,Reason for cancelling by Customer,Cancelled Rides by Driver,Driver Cancellation Reason,Incomplete Rides,Incomplete Rides Reason,Booking Value,Ride Distance,Driver Ratings,Customer Rating,Payment Method,datetime,hour,day_of_week,is_weekend,missing_driver_rating,missing_customer_rating,missing_booking_value,missing_payment_method
317,"""CNR5071968""",Completed,"""CID7384045""",Go Sedan,Panchsheel Park,Yamuna Bank,4.7,42.5,0,NaN,0,NaN,0,NaN,473.0,48.35,4.7,3.8,Cash,2024-10-10 03:56:19,3,Thursday,False,0,0,0,0
5522,"""CNR5071968""",No Driver Found,"""CID6309096""",Auto,Kanhaiya Nagar,India Gate,NaN,NaN,0,NaN,0,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,2024-03-10 19:55:06,19,Sunday,True,1,1,1,1


In [744]:
df[df['Booking ID']=='"CNR8512595"']

,Booking ID,Booking Status,Customer ID,Vehicle Type,Pickup Location,Drop Location,Avg VTAT,Avg CTAT,Cancelled Rides by Customer,Reason for cancelling by Customer,Cancelled Rides by Driver,Driver Cancellation Reason,Incomplete Rides,Incomplete Rides Reason,Booking Value,Ride Distance,Driver Ratings,Customer Rating,Payment Method,datetime,hour,day_of_week,is_weekend,missing_driver_rating,missing_customer_rating,missing_booking_value,missing_payment_method
1893,"""CNR8512595""",Completed,"""CID8017027""",Auto,Ashok Vihar,Mehrauli,14.6,29.6,0,NaN,0,NaN,0,NaN,294.0,44.22,3.6,5.0,Cash,2024-11-02 10:45:25,10,Saturday,True,0,0,0,0
7762,"""CNR8512595""",Completed,"""CID9741888""",Go Mini,Narsinghpur,Huda City Centre,3.5,15.4,0,NaN,0,NaN,0,NaN,187.0,41.45,4.4,4.2,Credit Card,2024-03-01 11:55:56,11,Friday,False,0,0,0,0


In [745]:
# create missing flags and impute Avg VTAT / Avg CTAT by median per Vehicle Type (fallback to global median)
df['Avg_VTAT_missing'] = df['Avg VTAT'].isnull().astype(int)
df['Avg_CTAT_missing'] = df['Avg CTAT'].isnull().astype(int)

# median per vehicle type, then fallback to overall median if vehicle-type median is NaN
vtat_med_by_type = df.groupby(['Vehicle Type','Pickup Location'])['Avg VTAT'].transform('median')
ctat_med_by_type = df.groupby(['Vehicle Type','Pickup Location'])['Avg CTAT'].transform('median')

df['Avg_VTAT_imputed'] = df['Avg VTAT'].fillna(vtat_med_by_type).fillna(df['Avg VTAT'].median())
df['Avg_CTAT_imputed'] = df['Avg CTAT'].fillna(ctat_med_by_type).fillna(df['Avg CTAT'].median())

# Optional: drop or keep original columns. Defaults keep both original and imputed.
# df.drop(['Avg VTAT','Avg CTAT'], axis=1, inplace=True)  # uncomment if you want to replace originals

In [746]:
df.head()

,Booking ID,Booking Status,Customer ID,Vehicle Type,Pickup Location,Drop Location,Avg VTAT,Avg CTAT,Cancelled Rides by Customer,Reason for cancelling by Customer,Cancelled Rides by Driver,Driver Cancellation Reason,Incomplete Rides,Incomplete Rides Reason,Booking Value,Ride Distance,Driver Ratings,Customer Rating,Payment Method,datetime,hour,day_of_week,is_weekend,missing_driver_rating,missing_customer_rating,missing_booking_value,missing_payment_method,Avg_VTAT_missing,Avg_CTAT_missing,Avg_VTAT_imputed,Avg_CTAT_imputed
0,"""CNR5884300""",No Driver Found,"""CID1982111""",eBike,Palam Vihar,Jhilmil,NaN,NaN,0,NaN,0,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,2024-03-23 12:29:38,12,Saturday,True,1,1,1,1,1,1,9.7,28.15
1,"""CNR1326809""",Incomplete,"""CID4604802""",Go Sedan,Shastri Nagar,Gurgaon Sector 56,4.9,14.0,0,NaN,0,NaN,1,Vehicle Breakdown,237.0,5.73,NaN,NaN,UPI,2024-11-29 18:01:39,18,Friday,False,1,1,0,0,0,0,4.9,14.00
2,"""CNR8494506""",Completed,"""CID9202816""",Auto,Khandsa,Malviya Nagar,13.4,25.8,0,NaN,0,NaN,0,NaN,627.0,13.58,4.9,4.9,Debit Card,2024-08-23 08:56:10,8,Friday,False,0,0,0,0,0,0,13.4,25.80
3,"""CNR8906825""",Completed,"""CID2610914""",Premier Sedan,Central Secretariat,Inderlok,13.1,28.5,0,NaN,0,NaN,0,NaN,416.0,34.02,4.6,5.0,UPI,2024-10-21 17:17:25,17,Monday,False,0,0,0,0,0,0,13.1,28.50
4,"""CNR1950162""",Completed,"""CID9933542""",Bike,Ghitorni Village,Khan Market,5.3,19.6,0,NaN,0,NaN,0,NaN,737.0,48.21,4.1,4.3,UPI,2024-09-16 22:08:00,22,Monday,False,0,0,0,0,0,0,5.3,19.60


In [747]:
# df1= df.drop('Booking ID', axis=1)
# df1.head()

In [748]:
df.duplicated().sum()  # check for duplicate rows

np.int64(0)

In [750]:
df.describe() 

,Avg VTAT,Avg CTAT,Cancelled Rides by Customer,Cancelled Rides by Driver,Incomplete Rides,Booking Value,Ride Distance,Driver Ratings,Customer Rating,datetime,hour,missing_driver_rating,missing_customer_rating,missing_booking_value,missing_payment_method,Avg_VTAT_missing,Avg_CTAT_missing,Avg_VTAT_imputed,Avg_CTAT_imputed
count,139500.000000,102000.000000,150000.000000,150000.000000,150000.000000,102000.000000,102000.000000,93000.000000,93000.000000,150000,150000.000000,150000.000000,150000.000000,150000.000000,150000.000000,150000.000000,150000.000000,150000.000000,150000.000000
mean,8.456352,29.149636,0.070000,0.180000,0.060000,508.295912,24.637012,4.230992,4.404584,2024-07-01 07:14:41.251033344,14.034113,0.380000,0.380000,0.320000,0.320000,0.070000,0.320000,8.442259,29.042137
min,2.000000,10.000000,0.000000,0.000000,0.000000,50.000000,1.000000,3.000000,3.000000,2024-01-01 00:19:34,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,2.000000,10.000000
25%,5.300000,21.600000,0.000000,0.000000,0.000000,234.000000,12.460000,4.100000,4.200000,2024-03-31 22:55:36.249999872,10.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,5.600000,24.900000
50%,8.300000,28.800000,0.000000,0.000000,0.000000,414.000000,23.720000,4.300000,4.500000,2024-07-01 09:24:52.500000,15.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,8.250000,28.800000
75%,11.300000,36.800000,0.000000,0.000000,0.000000,689.000000,36.820000,4.600000,4.800000,2024-09-30 13:46:07.249999872,18.000000,1.000000,1.000000,1.000000,1.000000,0.000000,1.000000,11.000000,33.100000
max,20.000000,45.000000,1.000000,1.000000,1.000000,4277.000000,50.000000,5.000000,5.000000,2024-12-30 23:36:11,23.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,20.000000,45.000000
std,3.773564,8.902577,0.255148,0.384189,0.237488,395.805774,14.002138,0.436871,0.437819,NaN,5.416906,0.485388,0.485388,0.466478,0.466478,0.255148,0.466478,3.642278,7.399978


## EDA

print('hello')